# 00 — Download & Prepare PickSense Data

**Run this notebook ONCE (or whenever the dataset changes).**

It will:
1. Mount Google Drive (permanent storage).
2. Download the OpenLORIS dataset **only if it is missing**.
3. Persist it to Drive so it survives Colab runtime restarts.
4. Build the small **PickSense Mini** dataset (300 images) **only if it is missing**.
5. Print a verification summary.

Your normal work happens in `01_picksense_main.ipynb`, which never downloads anything.

In [1]:
# Mount Google Drive so everything we create is permanent (survives restarts).
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# kagglehub is the same downloader used in the original PickSense notebook.
!pip install -q kagglehub

In [7]:
from pathlib import Path



# --- Permanent project location on Google Drive (NOT /content, which is temporary) ---

PROJECT_DIR = Path("/content/drive/MyDrive/PickSense")

RAW_DIR = PROJECT_DIR / "data" / "raw" / "openloris"

MINI_DIR = PROJECT_DIR / "data" / "picksense_mini"



# --- Balanced dataset settings ---

RANDOM_SEED = 42

TRAIN_PER_CLASS = 1000

TEST_PER_CLASS = 200

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

CLASSES = ["clear", "partially_occluded", "heavily_occluded"]



# --- OpenLORIS 9 tasks -> 3 pickability classes ---

TASK_TO_LABEL = {

    "task1": "clear", "task2": "clear", "task3": "clear",

    "task4": "partially_occluded", "task5": "partially_occluded", "task6": "partially_occluded",

    "task7": "heavily_occluded", "task8": "heavily_occluded", "task9": "heavily_occluded",

}



PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)

print("Raw dir    :", RAW_DIR)

print("Dataset dir:", MINI_DIR)

print(f"Requested images: {TRAIN_PER_CLASS * len(CLASSES)} train + "

      f"{TEST_PER_CLASS * len(CLASSES)} test")

Project dir: /content/drive/MyDrive/PickSense
Raw dir    : /content/drive/MyDrive/PickSense/data/raw/openloris
Dataset dir: /content/drive/MyDrive/PickSense/data/picksense_mini
Requested images: 3000 train + 600 test


## Step 1 — Download OpenLORIS (only if missing)

We check whether the `occlusion` folder already exists on Drive. If it does, we
skip the download entirely. Otherwise we download once with `kagglehub` and copy
the `occlusion` subset to Drive.

> The full OpenLORIS-Object dataset is very large (~40–50 GB). PickSense only uses
> the **occlusion** condition, so by default we persist just that subset to save
> Drive space. Set `PERSIST_ONLY_OCCLUSION = False` to keep the entire download.

In [4]:
import shutil
import kagglehub

PERSIST_ONLY_OCCLUSION = True  # keep only the 'occlusion' subset on Drive (saves space)

def find_occlusion(base: Path):
    # Return the first 'occlusion' folder found under `base`, or None.
    if not base.exists():
        return None
    for p in base.rglob("occlusion"):
        if p.is_dir():
            return p
    return None

occ = find_occlusion(RAW_DIR)

if occ is not None:
    # Requirement: do NOT download again if it is already present.
    print("Dataset already exists. Skipping download.")
    print("occlusion source:", occ)
else:
    print("Dataset not found. Downloading OpenLORIS from Kaggle (one-time)...")
    # Same downloader + handle as the original PickSense notebook.
    cache_path = Path(kagglehub.dataset_download("zhedamai/openlorisobject"))
    print("Downloaded to temporary cache:", cache_path)

    # kagglehub downloads to a TEMPORARY cache. Copy what we need onto Drive so it
    # survives runtime restarts.
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    if PERSIST_ONLY_OCCLUSION:
        src_occ = find_occlusion(cache_path)
        if src_occ is None:
            raise FileNotFoundError("Could not find an 'occlusion' folder in the download.")
        dest_occ = RAW_DIR / "occlusion"
        print("Copying occlusion subset to Drive:")
        print(" ", src_occ, "->", dest_occ)
        shutil.copytree(src_occ, dest_occ, dirs_exist_ok=True)
    else:
        print("Copying the full dataset to Drive:")
        print(" ", cache_path, "->", RAW_DIR)
        shutil.copytree(cache_path, RAW_DIR, dirs_exist_ok=True)

    occ = find_occlusion(RAW_DIR)
    print("Permanent occlusion path:", occ)

Dataset already exists. Skipping download.
occlusion source: /content/drive/MyDrive/PickSense/data/raw/openloris/occlusion


## Step 2 — Build PickSense Mini (only if missing)

We create a small, balanced 300-image dataset (75 per class for train, 25 per
class for test). Train images are sampled from `occlusion/train` and test images
from `occlusion/test`, so **no image is ever shared** between train and test.
Sampling uses `random.seed(42)` for reproducibility, and images are **copied**
(never moved), so your OpenLORIS data is never modified.

In [ ]:
import random

def count_images(folder: Path) -> int:
    # Count image files directly inside a folder.
    if not folder.exists():
        return 0
    return sum(1 for p in folder.iterdir()
               if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

def mini_is_complete() -> bool:
    # True only if every class already has the expected number of images.
    for split, need in [("train", TRAIN_PER_CLASS), ("test", TEST_PER_CLASS)]:
        for c in CLASSES:
            if count_images(MINI_DIR / split / c) < need:
                return False
    return True

def list_images_for_label(split_dir: Path, label: str):
    # All images under the tasks that map to `label`, sorted for reproducibility.
    files = []
    for task, lab in TASK_TO_LABEL.items():
        if lab != label:
            continue
        task_dir = split_dir / task
        if not task_dir.is_dir():
            continue
        for p in task_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
                files.append(p)
    files.sort()
    return files

def unique_name(src: Path, base: Path) -> str:
    # Flatten 'task1/obj/frame.jpg' -> 'task1_obj_frame.jpg' to avoid name clashes.
    return "_".join(src.relative_to(base).parts)

if mini_is_complete():
    print("PickSense Mini already exists.")
    print("Skipping dataset creation.")
else:
    occ = find_occlusion(RAW_DIR)
    if occ is None:
        raise FileNotFoundError("OpenLORIS 'occlusion' not found. Re-run Step 1 first.")

    train_src, test_src = occ / "train", occ / "test"

    # Fresh build so counts are exact. Safety check: only ever delete the mini folder.
    if MINI_DIR.exists():
        assert MINI_DIR.name == "picksense_mini", "Refusing to delete a non-mini folder."
        shutil.rmtree(MINI_DIR)

    random.seed(RANDOM_SEED)  # reproducible selection

    for split, src_base, need in [("train", train_src, TRAIN_PER_CLASS),
                                  ("test", test_src, TEST_PER_CLASS)]:
        for c in CLASSES:
            files = list_images_for_label(src_base, c)
            if len(files) < need:
                # Do not crash: warn and copy whatever is available.
                print("WARNING:", split + "/" + c, "has only", len(files),
                      "images but", need, "are needed. Copying all available.")
                chosen = files
            else:
                chosen = random.sample(files, need)

            out_dir = MINI_DIR / split / c
            out_dir.mkdir(parents=True, exist_ok=True)
            for src in chosen:
                shutil.copy2(src, out_dir / unique_name(src, src_base))  # copy, never move

    print("PickSense Mini created.")

## Step 3 — Verification summary

In [6]:
def dir_size_bytes(path: Path) -> int:
    total = 0
    if path.exists():
        for p in path.rglob("*"):
            if p.is_file():
                total += p.stat().st_size
    return total

def human_size(num_bytes: int) -> str:
    size = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size < 1024:
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} PB"

occ = find_occlusion(RAW_DIR)

print("=" * 46)
print("PickSense data verification")
print("=" * 46)
print("Raw OpenLORIS path :", RAW_DIR)
print("OpenLORIS exists   :", occ is not None)
if occ is not None:
    print("occlusion folder   :", occ)
print("PickSense Mini path:", MINI_DIR)
print()

grand_total = 0
for split, need in [("train", TRAIN_PER_CLASS), ("test", TEST_PER_CLASS)]:
    print(split.capitalize() + ":")
    split_total = 0
    for c in CLASSES:
        n = count_images(MINI_DIR / split / c)
        print("  " + c + ":", n)
        split_total += n
    print("  Total " + split + ":", split_total)
    grand_total += split_total
    print()

print("Total dataset:", grand_total, "images")
print("Total disk size:", human_size(dir_size_bytes(MINI_DIR)))

PickSense data verification
Raw OpenLORIS path : /content/drive/MyDrive/PickSense/data/raw/openloris
OpenLORIS exists   : True
occlusion folder   : /content/drive/MyDrive/PickSense/data/raw/openloris/occlusion
PickSense Mini path: /content/drive/MyDrive/PickSense/data/picksense_mini

Train:
  clear: 120
  partially_occluded: 120
  heavily_occluded: 120
  Total train: 360

Test:
  clear: 30
  partially_occluded: 30
  heavily_occluded: 30
  Total test: 90

Total dataset: 450 images
Total disk size: 14.1 MB
